# OSRT Ostinato — v7 midtrain on Colab

Target runtime: **RTX PRO 6000 Blackwell, 96GB**.

Run the cells in order. Cell 1 is a gate — if it does not report `sm_120`,
stop and read its verdict before spending anything.

**Sessions are capped and the VM's disk is wiped on release.** Cell 5 sets
`--hf-repo`, which pulls the newest checkpoint before training and pushes
each new one as it is written. Without it a disconnect loses the run.

Do **not** background the training cell (`nohup`, `&`). Colab tears down the
runtime when the foreground cell returns, so a backgrounded trainer dies
with it — run it in the foreground and leave the tab open.


## 1 · Gate: what card is this, and can the expert path go low-precision?

This is also the first half of roadmap gate **G7**. Routed experts are ~84%
of v7's parameters and run through the private `torch._grouped_mm`; if that
refuses FP8, the NVFP4 case in §13.3 covers under a third of the model.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv


## 2 · Install

The resolver's output is left visible on purpose — a silent downgrade here is
how a run dies an hour in. The gate below refuses **torch < 2.10** or
**transformers < 5.3** (the pins `app.py`'s Modal image carries): torch 2.8 is
known to fail Inductor's C++ codegen on this model's compiled forward.


In [ ]:
!git clone -q https://github.com/CodeHalwell/OSRT-Ostinato.git
%cd OSRT-Ostinato
# Same pins as the Modal image in app.py. Colab's preinstalled torch and
# transformers satisfy pyproject's floors, so a bare `pip install -e .` would
# leave them in place and the version check in the next cell would abort.
!pip install -q torch==2.10.0+cu128 --index-url https://download.pytorch.org/whl/cu128
# Colab's preinstalled torchvision/torchaudio were built against ITS torch; left in
# place they break `import transformers` (torchvision::nms does not exist).
!pip uninstall -q -y torchvision torchaudio
!pip install -q transformers==5.3.0 datasets==4.6.1 tokenizers==0.22.2
!pip install -q safetensors==0.7.0 "huggingface_hub>=0.35" wandb==0.25.1
!pip install -q -e .
!PYTHONPATH=src python scripts/probe_gpu.py


In [ ]:
# Same floor as the Modal image in app.py (torch 2.10.0+cu128, transformers
# 5.3.0). torch 2.8 dies in Inductor codegen ("'zuf3' was not declared in this
# scope") at the first compiled forward, so refuse before spending anything.
import torch
import transformers
from packaging.version import Version

assert Version(torch.__version__.split('+')[0]) >= Version('2.10'), torch.__version__
assert Version(transformers.__version__) >= Version('5.3'), transformers.__version__
print('torch', torch.__version__, '| transformers', transformers.__version__)


## 3 · Secrets

Add `HF_TOKEN` (write access to your checkpoint repo) and `WANDB_API_KEY`
in the Colab **Secrets** panel (🔑, left sidebar). They never touch this
notebook or git.


In [ ]:
import os

from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert os.environ['HF_TOKEN'] and os.environ['WANDB_API_KEY'], 'set both in Secrets'
print('secrets set')


## 4 · Check the tokenizer and confirm the shape

The v7 tokenizer is **committed** in `tokenizer/` (SmolLM2 + 32 OSRT specials,
roadmap §16). Do not rebuild it per session: a rebuild that drifted by one
token would silently invalidate every checkpoint. The contract check pins the
vocab size and every structural token id, and is the same check the trainer
runs before building the model.

`compute_budget.py` is the only trusted source for parameter counts — the
repo deliberately states none in any name. Expect **968,468,355 physical /
263,035,779 active**; anything else means the config drifted.


In [ ]:
%%bash
# Tokenizer contract + the canonical parameter counts, before any GPU spend.
PYTHONPATH=src python - <<'EOF'
from transformers import AutoTokenizer
from osrt.tokenizer_contract import validate_tokenizer_contract
validate_tokenizer_contract(AutoTokenizer.from_pretrained('tokenizer'))
print('tokenizer contract OK')
EOF
PYTHONPATH=src python scripts/compute_budget.py


## 5 · The run

This continues **midtrain v4** (`MidtrainConfig`: WSD, peak 5e-5, the widened
24-source mix — docs/PROGRESS.md §9f) from wherever the Modal chain left it.
`--config midtrain --hf-repo HallD/OSRT-Ostinato-midtrain-v4` pulls the newest
`osrt_step_N.pt` (weights + optimizer + data cursors) from the private mirror
and resumes; every new checkpoint is pushed back, so a Modal workspace with
credit can take over again later with its usual launch command.

Leave the tab open. Do **not** background the cell. When the session dies —
it will — re-run this cell; it resumes from the mirror.

> **Card size.** The recipe's 270,336 tokens/step are 6 × 4096 × 11 on a
> B200; `--micro-batch-scale 0.5` makes that 3 × 4096 × 22 on the 96 GB RTX
> PRO 6000 — same tokens per step, same run (the resume check does not key on
> micro-batch shape). Expect ~15–20K tok/s against the B200's 42K.

> **One W&B run.** The run id travels in `wandb_run_id.txt` on the mirror, so
> this continues W&B run `osrt-v7-midtrain-v4`. The dashboard may show the run
> as "crashed" right after a relay — the previous process never finished;
> trust new rows, not the label.


In [ ]:
# sm_120 (this card) caps shared memory at 99 KB/block; Inductor's persistent
# fused-RMSNorm backward kernel needs ~120 KB. Read by torch at import, so it
# must be in the environment of the training process, not set inside it.
%env TORCHINDUCTOR_PERSISTENT_REDUCTIONS=0
HF_CKPT_REPO = 'HallD/OSRT-Ostinato-midtrain-v4'   # private mirror of the v4 run

!PYTHONPATH=src python -m osrt.train_main \
    --config midtrain \
    --tokenizer-path ./tokenizer \
    --ckpt-dir ./checkpoints/midtrain_v4 \
    --hf-repo {HF_CKPT_REPO} \
    --micro-batch-scale 0.5


## 6 · What to watch

The health check runs on **every log step** (no longer only at step 5,000)
and ends the run on router collapse, loop collapse, residual explosion or dead
data — that is the design. The last line of the cell names the outcome:
`early_stop` (the log names the criterion; `osrt_failed_step_N.pt` holds the
state and is not resumed), `rescued` (time boundary — re-run the cell),
`data_dead` (every source failed for good — fix the data, re-run), `complete`.
Otherwise:

| signal | healthy | worry |
|---|---|---|
| `loss` | falling, no unrecovered spikes | flat, or spiking |
| `train/grad_norm` | settles after warmup, spikes recover | climbing, or a staircase |
| `moe/dead_experts_total` | 0 | > 0 — at E=28 each is 3.6% of a block |
| `moe/marginal_entropy_mean` | near ln(28) ≈ 3.33 | falling — router collapse |
| `moe/prebias_expert_min_mean` | clearly > 0 | → 0 while loss still falls: the balance bias is carrying a dead router |
| `moe/b*/bias_loop_spread` | flat | rising — routing diverging across loops |
| `loop/update_norm_l*` | non-trivial at every loop | last loops → 0 = loop collapse |
| `loop/hidden_norm_ratio` | ≈ 1, flat | rising — residual explosion (§17.3) |
| `muon/ortho_err` | small, flat | rising — Newton–Schulz not converging |
| `train/dead_sources` | 0 | > 0 — a data source failed permanently |

The bets this run embodies, and what would falsify each, are in roadmap §19.


## 7 · Cooldown branch (when the lineage reaches the token target)

A WSD plateau checkpoint is usable as-is, but a short decay to `min_lr` is
the cheap, standard way to get a better release/SFT base without touching
the main run. This branches off the newest `osrt_step_N.pt`: the optimizer
state is kept, the run ends `COOLDOWN_STEPS` later, LR decays linearly
5e-5 → 5e-6 over exactly those steps (`train_main --cooldown-steps`,
tests/test_cooldown_and_soup.py). Evals land every 500 steps as usual.

It pushes to a **separate** mirror so the main run's "latest" stays the
plateau checkpoint and a Modal workspace can resume the plateau later. Set
`FROM_STEP` to the plateau checkpoint you want to branch from (it is pulled
from the main mirror first).


In [ ]:
# sm_120 (this card) caps shared memory at 99 KB/block; Inductor's persistent
# fused-RMSNorm backward kernel needs ~120 KB. Read by torch at import, so it
# must be in the environment of the training process, not set inside it.
%env TORCHINDUCTOR_PERSISTENT_REDUCTIONS=0
FROM_STEP = 13500                                  # <-- plateau checkpoint to branch from
COOLDOWN_STEPS = 1000
COOLDOWN_REPO = 'HallD/OSRT-Ostinato-midtrain-v4-cooldown'   # private; created on first push

from huggingface_hub import hf_hub_download
hf_hub_download('HallD/OSRT-Ostinato-midtrain-v4', f'osrt_step_{FROM_STEP}.pt',
                local_dir='./checkpoints/cooldown')

!PYTHONPATH=src python -m osrt.train_main \
    --config midtrain \
    --tokenizer-path ./tokenizer \
    --ckpt-dir ./checkpoints/cooldown \
    --hf-repo {COOLDOWN_REPO} \
    --cooldown-steps {COOLDOWN_STEPS} \
    --micro-batch-scale 0.5 \
    --wandb-run-name osrt-v7-midtrain-v4-cooldown


## 8 · Soup

The trunk's 3-checkpoint soup beat every member (PROGRESS §9b). This scores
each listed checkpoint and their fp32 mean on the same held-out batches
(4096 × 6, 20 steps) and saves `osrt_soup_<steps>.pt` (weights only).
Candidates: the last three plateau checkpoints, or the last three of the
cooldown branch (`--repo` = the cooldown mirror). `--push` uploads the soup
under `soup/` in that repo.


In [ ]:
SOUP_REPO = 'HallD/OSRT-Ostinato-midtrain-v4-cooldown'
SOUP_CKPTS = 'osrt_step_13500.pt,osrt_step_14000.pt,osrt_step_14500.pt'

!PYTHONPATH=src python scripts/soup.py --repo {SOUP_REPO} --ckpts {SOUP_CKPTS} --push


## 9 · SFT probe

`SFTProbeConfig` (docs/specs/2026-10-03-sft-probe.md): HRA rank-256 adapters
on the FROZEN base, assistant-only loss, 1,000 steps ≈ 0.26B tokens of
Dolci / smoltalk2 / OpenCodeInstruct / OpenMathInstruct-2. The probe shape
(seq 4096, batch 8 × accum 8) ran on an 80 GB H100, so it fits here as-is.
Prints the held-out SFT loss and fineweb loss at step 0 (= the base) and
every 250 steps; merges the adapters and, with `--push`, uploads
`sft/<run>_merged_step_1000.pt` + bf16 safetensors to `--base-repo`.

Baseline to beat (same probe on the trunk soup): GSM8K 7.0 strict / 10.5
flexible, HumanEval 9.8% pass@1 — scored on Modal with
`scripts/lm_eval_trunk.py --chat --fast` once a workspace has credit.


In [ ]:
BASE_REPO = 'HallD/OSRT-Ostinato-midtrain-v4-cooldown'
BASE_FILE = 'soup/osrt_soup_13500_14000_14500.pt'   # or an osrt_step_N.pt

!PYTHONPATH=src python scripts/run_sft.py \
    --base-repo {BASE_REPO} --base-file {BASE_FILE} \
    --run-name osrt-v7-sft-v4 --push
